In [1]:
import os
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
from PIL import Image
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import pandas as pd
device = torch.device('cuda:2') if torch.cuda.is_available() else torch.device('cpu')


#### Dataset Class


In [2]:
class VOCTextDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.image_files = [f for f in os.listdir(images_dir) if f.endswith('.tif')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_filename = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_filename)
        label_path = os.path.join(self.labels_dir, img_filename.replace('.tif', '.txt'))

        # Load image
        img = Image.open(img_path).convert("RGB")

        # Parse annotations
        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        bbox = list(map(float, parts[1:5]))
                        boxes.append(bbox)
                        labels.append(cls_id)

        # Safe fallback if label file missing or empty
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx])
        }

        if self.transforms:
            img = self.transforms(img)
            

        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

def get_transform():
    return torchvision.transforms.Compose([
        torchvision.transforms.ToTensor()
    ])




# --- Prediction + Evaluation ---

In [3]:

def predict(model, test_image_dir, transform, prediction_output_dir, batch_size=128):
    os.makedirs(prediction_output_dir, exist_ok=True)
    model.eval()

    # Collect all .tif filenames
    image_files = sorted([f for f in os.listdir(test_image_dir) if f.lower().endswith(".tif")])

    for i in tqdm(range(0, len(image_files), batch_size)):
        batch_files = image_files[i:i + batch_size]
        images = []
        original_filenames = []

        for img_file in batch_files:
            img_path = os.path.join(test_image_dir, img_file)
            image = Image.open(img_path).convert("RGB")
            image_tensor = transform(image)
            images.append(image_tensor)
            original_filenames.append(img_file)

        # Send to device
        images = [img.to(device) for img in images]

        with torch.no_grad():
            batch_outputs = model(images)

        # Save each prediction to its corresponding file
        for img_file, outputs in zip(original_filenames, batch_outputs):
            base_name = os.path.splitext(img_file)[0]
            pred_txt_path = os.path.join(prediction_output_dir, base_name + ".txt")
            with open(pred_txt_path, "w") as f:
                for box, label, score in zip(outputs["boxes"], outputs["labels"], outputs["scores"]):
                    x1, y1, x2, y2 = box.tolist()
                    f.write(f"{label.item()} {score.item():.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}\n")


def load_txt_as_ground_truth(txt_path):
    boxes, labels = [], []
    if os.path.exists(txt_path):
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls, x1, y1, x2, y2 = map(float, parts)
                labels.append(int(cls)+1)
                boxes.append([x1, y1, x2, y2])
    return {
        "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
        "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,))
    }

def load_txt_as_prediction(txt_path):
    boxes, labels, scores = [], [], []
    if os.path.exists(txt_path):
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 6:
                    continue
                cls, conf, x1, y1, x2, y2 = map(float, parts)
                labels.append(int(cls)+1)
                scores.append(conf)
                boxes.append([x1, y1, x2, y2])
    return {
        "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
        "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,)),
        "scores": torch.tensor(scores, dtype=torch.float32) if scores else torch.zeros((0,))
    }

def make_class_agnostic(detection_list):
    for d in detection_list:
        d["labels"] = torch.zeros_like(d["labels"])
    return detection_list

def compute_map(test_img_dir, test_label_dir, pred_dir):
    image_filenames = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(".tif")])
    gt_targets, predictions = [], []
    for fname in image_filenames:
        base = os.path.splitext(fname)[0]
        gt = load_txt_as_ground_truth(os.path.join(test_label_dir, base + ".txt"))
        pred = load_txt_as_prediction(os.path.join(pred_dir, base + ".txt"))
        gt_targets.append(gt)
        predictions.append(pred)
    gt_targets = make_class_agnostic(gt_targets)
    predictions = make_class_agnostic(predictions)
    metric = MeanAveragePrecision()
    metric.update(predictions, gt_targets)
    results = metric.compute()
    return results["map_50"].item()

In [4]:
base_dir = "/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/domain_experiment/data"
images_dir=f"{base_dir}/delhi_airshed/images"
labels_dir=f"{base_dir}/delhi_airshed/labels_voc"
out_region_images = f"{base_dir}/lucknow_airshed_100/images"
out_region_labels = f"{base_dir}/lucknow_airshed_100/labels_voc"

in_region_images = f"{base_dir}/test_delhi_airshed/images"
in_region_labels = f"{base_dir}/test_delhi_airshed/labels_voc"
output_root = "./inference_outputs1"
os.makedirs(output_root, exist_ok=True)


In [5]:
transform = GeneralizedRCNNTransform(
    min_size=640,
    max_size=640,
    image_mean=[0.485, 0.456, 0.406],
    image_std=[0.229, 0.224, 0.225]
)

In [6]:
model = fasterrcnn_resnet50_fpn(num_classes=4)  
model.transform = transform
model.to(device)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640,), max_size=640, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
          (relu)

In [7]:
# Load data
train_dataset = VOCTextDataset(images_dir, labels_dir, get_transform())
train_loader = DataLoader(train_dataset, batch_size=148, shuffle=True, collate_fn=collate_fn)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)

# Training loop with evaluation every 5 epochs
csv_path = "map_logs.csv"
log_df = pd.DataFrame(columns=["epoch", "map50_outregion", "map50_inregion"])

for epoch in range(1, 101):
    model.train()
    total_loss = 0
    for imgs, targets in tqdm(train_loader, desc=f"Epoch {epoch}"):
        imgs = list(img.to(device) for img in imgs)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()

    print(f"Epoch {epoch}: Total Loss = {total_loss:.4f}")
    # Save model checkpoint
    if epoch % 10 == 0:
        checkpoint_path = f"/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/domain_experiment/an_embedding_active_learning/baselines_weight/faster_rcnn_epoch_{epoch}.pth"
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Model saved to {checkpoint_path}")

    # --- Inference + Evaluation ---
    if epoch % 10 == 0:
        pred_out = os.path.join(output_root, f"epoch_{epoch}_outregion")
        pred_in = os.path.join(output_root, f"epoch_{epoch}_inregion")

        predict(model, out_region_images, get_transform(), pred_out)
        predict(model, in_region_images, get_transform(), pred_in)

        map_out = compute_map(out_region_images, out_region_labels, pred_out)
        map_in = compute_map(in_region_images, in_region_labels, pred_in)

        print(f"[Epoch {epoch}] mAP@0.50 (Out-region): {map_out:.4f} | (In-region): {map_in:.4f}")

        log_df.loc[len(log_df)] = [epoch, map_out, map_in]
        log_df.to_csv(csv_path, index=False)


Epoch 1:  62%|██████▎   | 5/8 [00:45<00:27,  9.18s/it]


KeyboardInterrupt: 